# ⚠️ LEGACY / NON-THESIS EVALUATION

This notebook preserves a historical or diagnostic protocol and **must not be used for cross-model thesis comparisons**. Use the authoritative notebook in `notebooks/Transformer-JEPA/othello_gpt_jepa_smoke_evaluation.ipynb` or the appropriate `notebooks/12x12/*_evaluation.ipynb` frontend for `unified_eval_v1`.

# Othello JEPA Smoke Evaluation

Compact evaluation of a frozen JEPA context encoder.

This notebook runs only four downstream evaluations:

1. Linear next-move head, evaluated by legal-move metrics.
2. MLP next-move head, evaluated by legal-move metrics.
3. Linear board-state probes at every encoder layer, for absolute and relative labels.
4. MLP board-state probes at every encoder layer, for absolute and relative labels.

All downstream models train on the first **20 pretraining chunks**. The JEPA encoder is never fine-tuned.
Artifacts and a consolidated Markdown summary are written to `<run>/smoke_eval/`.

For the next-move heads, `top-1 legal` is the fraction of argmax predictions that are legal.
`top-3/5 legal fraction` is the mean fraction of the top-k predictions that are legal.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure Project, Data, Run, And Smoke Budget

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import yaml

PROJECT_ROOT = Path('/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/Master_Thesis_Artifacts')
DATA_ROOT = ARTIFACT_ROOT / 'data'
RUNS_DIR = ARTIFACT_ROOT / 'runs'
CONFIGS_DIR = PROJECT_ROOT / 'configs'

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert (PROJECT_ROOT / 'pyproject.toml').exists(), PROJECT_ROOT
assert (PROJECT_ROOT / 'othello_research').is_dir(), PROJECT_ROOT
assert DATA_ROOT.exists(), DATA_ROOT
assert RUNS_DIR.exists(), RUNS_DIR
assert CONFIGS_DIR.exists(), CONFIGS_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)

# === SINGLE SOURCE OF TRUTH ===
# Change only this filename to evaluate another JEPA experiment.
CONFIG_NAME = 'family_infonce_v2_cpf05.yml'
CHECKPOINT_NAME = 'final.pt'

# Every downstream head/probe trains on exactly this many pretraining chunks.
EVAL_TRAIN_CHUNKS = 20

CONFIG_PATH = CONFIGS_DIR / CONFIG_NAME
assert CONFIG_PATH.exists(), CONFIG_PATH
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    EXPERIMENT_CONFIG = yaml.safe_load(f)

def config_value(section, key, default=None):
    section_data = EXPERIMENT_CONFIG.get(section, {})
    if isinstance(section_data, dict) and key in section_data:
        return section_data[key]
    return EXPERIMENT_CONFIG.get(key, default)

BOARD_SIZE = int(config_value('data', 'board_size', 8))
JEPA_RUN_NAME = str(config_value('experiment', 'name'))
N_TRAIN_CHUNKS = int(config_value('data', 'li_split_train_chunks', 200))

assert JEPA_RUN_NAME and JEPA_RUN_NAME != 'None', 'Missing experiment.name in YAML'
assert EVAL_TRAIN_CHUNKS <= N_TRAIN_CHUNKS, (
    f'EVAL_TRAIN_CHUNKS={EVAL_TRAIN_CHUNKS} exceeds this run pretraining split '
    f'of {N_TRAIN_CHUNKS} chunks.'
)

DATA_DIR = DATA_ROOT / f'othello_{BOARD_SIZE}x{BOARD_SIZE}'
JEPA_OUT_DIR = RUNS_DIR / JEPA_RUN_NAME
SMOKE_EVAL_DIR = JEPA_OUT_DIR / 'smoke_eval'
LOCAL_SMOKE_DIR = Path('/content/jepa_smoke_eval') / JEPA_RUN_NAME
SMOKE_EVAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_SMOKE_DIR.mkdir(parents=True, exist_ok=True)

all_chunks = sorted(DATA_DIR.glob('*.pickle'))
assert len(all_chunks) > N_TRAIN_CHUNKS, f'Need held-out validation chunks in {DATA_DIR}'
train_chunks = all_chunks[:EVAL_TRAIN_CHUNKS]
val_chunks = all_chunks[N_TRAIN_CHUNKS:]
assert len(train_chunks) == EVAL_TRAIN_CHUNKS
assert val_chunks

print('CONFIG_NAME       :', CONFIG_NAME)
print('CONFIG_PATH       :', CONFIG_PATH)
print('JEPA_RUN_NAME     :', JEPA_RUN_NAME)
print('BOARD_SIZE        :', BOARD_SIZE)
print('N_TRAIN_CHUNKS    :', N_TRAIN_CHUNKS)
print('EVAL_TRAIN_CHUNKS :', len(train_chunks))
print('VAL_CHUNKS        :', len(val_chunks))
print('DATA_DIR          :', DATA_DIR)
print('JEPA_OUT_DIR      :', JEPA_OUT_DIR)
print('SMOKE_EVAL_DIR    :', SMOKE_EVAL_DIR)

## 3. Select Checkpoint

In [ ]:
checkpoint_files = sorted(JEPA_OUT_DIR.glob('*.pt'))
print('Available checkpoints:')
for path in checkpoint_files:
    print(' ', path.name)

DRIVE_CKPT = JEPA_OUT_DIR / CHECKPOINT_NAME
if not DRIVE_CKPT.exists():
    assert checkpoint_files, f'No checkpoint files found in {JEPA_OUT_DIR}'
    DRIVE_CKPT = checkpoint_files[-1]
    print(f'{CHECKPOINT_NAME} was not found; selected {DRIVE_CKPT.name}')
else:
    print('Selected:', DRIVE_CKPT)

## 4. Load And Freeze JEPA

In [ ]:
import gc
import importlib
import json
import math
import shutil
import time
from collections import defaultdict
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from IPython.display import Markdown, display

from othello_research.datasets.dataset import TARGET_PAD, OthelloChunkDataset, load_chunk
from othello_research.evaluation.legal_moves import evaluate_legal_moves
from othello_research.probes.board_state import (
    ActivationCache,
    LABEL_PAD,
    _iter_probe_batches,
    load_games_from_chunks,
)
import othello_research.training.train_jepa as tjm

importlib.reload(tjm)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOCAL_CKPT = LOCAL_SMOKE_DIR / f'jepa_{DRIVE_CKPT.stem}.pt'
if not LOCAL_CKPT.exists():
    print(f'Restoring {DRIVE_CKPT.stat().st_size / 1e6:.1f} MB from Drive...')
    shutil.copy2(DRIVE_CKPT, LOCAL_CKPT)
    print('done')

ckpt = torch.load(LOCAL_CKPT, map_location=device)
jepa_model, jepa_cfg = tjm.build_model_from_checkpoint(ckpt)
jepa_model.load_state_dict(ckpt['model'])
jepa_model.to(device)
jepa_model.eval()
for parameter in jepa_model.parameters():
    parameter.requires_grad_(False)

context_encoder = jepa_model.context_encoder
context_encoder.eval()
for parameter in context_encoder.parameters():
    parameter.requires_grad_(False)

PROBE_LAYERS = tuple(range(len(context_encoder.blocks) + 1))

objective_name = ckpt.get('model_config', {}).get(
    'objective_class',
    ckpt.get('train_config', {}).get('objective_class', 'jepa'),
)

SMOKE_RESULTS = {
    'metadata': {
        'config_name': CONFIG_NAME,
        'config_path': str(CONFIG_PATH),
        'run_name': JEPA_RUN_NAME,
        'run_dir': str(JEPA_OUT_DIR),
        'checkpoint_name': DRIVE_CKPT.name,
        'checkpoint_path': str(DRIVE_CKPT),
        'checkpoint_stem': DRIVE_CKPT.stem,
        'step': ckpt.get('step'),
        'games_seen': ckpt.get('games_seen', 0),
        'objective_class': objective_name,
        'eval_train_chunks': EVAL_TRAIN_CHUNKS,
        'train_chunks': [str(path) for path in train_chunks],
        'val_chunks': [str(path) for path in val_chunks],
        'layers': list(PROBE_LAYERS),
    },
    'next_move_legal': {},
    'board_state': {},
}

print('device         :', device)
print('checkpoint     :', DRIVE_CKPT.name)
print('step           :', ckpt.get('step'))
print('games seen     :', f"{ckpt.get('games_seen', 0):,}")
print('objective      :', objective_name)
print('context params :', f"{sum(p.numel() for p in context_encoder.parameters()):,}")
print('probe layers   :', PROBE_LAYERS)
print('JEPA frozen    :', not any(p.requires_grad for p in jepa_model.parameters()))

## 5. Evaluation Settings

In [ ]:
# Next-move heads: same optimizer family and basic budget as the existing evaluation.
HEAD_BATCH_SIZE = 512
HEAD_LR = 1e-3
HEAD_WEIGHT_DECAY = 0.0
HEAD_EPOCHS = 1
HEAD_PRECISION = 'bf16'

# Legal evaluation uses held-out games only.
LEGAL_EVAL_GAMES = 5000
LEGAL_K_VALUES = (1, 3, 5)
LEGAL_EVAL_BATCH_SIZE = 64

# Small nonlinear readout, not a second model.
MLP_HIDDEN_DIM = 512
MLP_N_LAYERS = 1
MLP_DROPOUT = 0.1

# Board probes sample from each of the 20 training chunks.
# This keeps the smoke eval broad enough to cover the split without
# accidentally training on millions of games.
PROBE_TRAIN_GAMES_PER_CHUNK = 250
PROBE_BATCH_SIZE = 64
PROBE_LR = 1e-3
PROBE_WEIGHT_DECAY = 0.0
PROBE_EPOCHS = 1
PROBE_VAL_GAMES = 512

print({
    'head_train_chunks': len(train_chunks),
    'head_epochs': HEAD_EPOCHS,
    'legal_eval_games': LEGAL_EVAL_GAMES,
    'probe_train_chunks': len(train_chunks),
    'probe_train_games_per_chunk': PROBE_TRAIN_GAMES_PER_CHUNK,
    'probe_train_games_total': PROBE_TRAIN_GAMES_PER_CHUNK * len(train_chunks),
    'probe_epochs': PROBE_EPOCHS,
    'probe_val_games': PROBE_VAL_GAMES,
    'mlp_hidden_dim': MLP_HIDDEN_DIM,
    'mlp_n_layers': MLP_N_LAYERS,
    'mlp_dropout': MLP_DROPOUT,
})

## 6. Shared Helpers

In [ ]:
def jsonable(obj):
    if isinstance(obj, dict):
        return {str(key): jsonable(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(value) for value in obj]
    if isinstance(obj, float) and math.isnan(obj):
        return None
    return obj


def amp_context(device, precision='bf16'):
    if device.type == 'cuda' and precision in {'bf16', 'fp16'}:
        dtype = torch.bfloat16 if precision == 'bf16' else torch.float16
        return torch.autocast(device_type='cuda', dtype=dtype, enabled=True)
    return nullcontext()


class MLPProbe(nn.Module):
    def __init__(self, d_in, d_out, hidden_dim=512, n_layers=1, dropout=0.1):
        super().__init__()
        if n_layers < 1:
            raise ValueError('n_layers must be >= 1')
        layers = [nn.Linear(d_in, hidden_dim), nn.GELU(), nn.Dropout(dropout)]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout)]
        layers += [nn.Linear(hidden_dim, d_out)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class FrozenJEPANextMoveHead(nn.Module):
    """Frozen context encoder plus a trainable linear or MLP next-move head."""

    def __init__(self, jepa_model, head_type):
        super().__init__()
        self.encoder = jepa_model.context_encoder
        self.encoder_config = jepa_model.config
        self.head_type = head_type
        for parameter in self.encoder.parameters():
            parameter.requires_grad_(False)
        self.encoder.eval()

        d_model = int(self.encoder_config.d_model)
        vocab_size = int(self.encoder_config.vocab_size)
        if head_type == 'linear':
            self.head = nn.Linear(d_model, vocab_size)
        elif head_type == 'mlp':
            self.head = MLPProbe(
                d_model,
                vocab_size,
                hidden_dim=MLP_HIDDEN_DIM,
                n_layers=MLP_N_LAYERS,
                dropout=MLP_DROPOUT,
            )
        else:
            raise ValueError(f'Unknown head_type: {head_type}')

    @property
    def config(self):
        return self.encoder_config

    def train(self, mode=True):
        super().train(mode)
        self.encoder.eval()
        return self

    def forward(self, idx, targets=None):
        if idx.ndim != 2:
            raise ValueError(f'Expected idx shape (B, T), got {tuple(idx.shape)}')
        _, seq_len = idx.shape
        if seq_len > self.config.block_size:
            raise ValueError(f'Input length {seq_len} exceeds block_size {self.config.block_size}')

        with torch.no_grad():
            enc = self.encoder
            pos = torch.arange(seq_len, device=idx.device)
            x = enc.drop(enc.wte(idx) + enc.wpe(pos))
            for block in enc.blocks:
                x = block(x)
            x = enc.ln_f(x)

        logits = self.head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=TARGET_PAD,
        )
        return logits, loss


def make_next_move_loader(chunk_path, model, *, shuffle):
    games = load_chunk(str(chunk_path))
    dataset = OthelloChunkDataset(
        games=games,
        block_size=model.config.block_size,
        board_size=BOARD_SIZE,
    )
    loader = DataLoader(
        dataset,
        batch_size=HEAD_BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    return loader, games, dataset


def train_next_move_head(head_type):
    model = FrozenJEPANextMoveHead(jepa_model, head_type=head_type).to(device)
    optimizer = torch.optim.AdamW(
        model.head.parameters(),
        lr=HEAD_LR,
        weight_decay=HEAD_WEIGHT_DECAY,
    )
    history = []

    for epoch in range(1, HEAD_EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        token_count = 0
        outer = tqdm(train_chunks, desc=f'{head_type} head epoch {epoch}/{HEAD_EPOCHS}', unit='chunk')
        for chunk_path in outer:
            loader, games, dataset = make_next_move_loader(chunk_path, model, shuffle=True)
            for x_cpu, y_cpu in tqdm(loader, desc=Path(chunk_path).name, leave=False):
                x = x_cpu.to(device, non_blocking=device.type == 'cuda')
                y = y_cpu.to(device, non_blocking=device.type == 'cuda')
                optimizer.zero_grad(set_to_none=True)
                with amp_context(device, HEAD_PRECISION):
                    logits, loss = model(x, y)
                loss.backward()
                optimizer.step()

                with torch.no_grad():
                    n_tokens = int((y != TARGET_PAD).sum().item())
                    loss_sum += float(loss.detach().cpu()) * n_tokens
                    token_count += n_tokens

            del loader, games, dataset
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            outer.set_postfix(loss=f'{loss_sum / max(1, token_count):.4f}')

        epoch_result = {
            'epoch': epoch,
            'train_loss': loss_sum / max(1, token_count),
            'train_tokens': token_count,
        }
        history.append(epoch_result)
        print(f"{head_type} head epoch={epoch} train_loss={epoch_result['train_loss']:.4f}")

    return model, history


def compact_legal_metrics(metrics):
    topk = metrics['topk_legal']
    return {
        'n_games': metrics['n_games'],
        'n_tokens': metrics['n_tokens'],
        'top1_legal_per_token': metrics['top1_legal_per_token'],
        'top3_legal_fraction': topk[3],
        'top5_legal_fraction': topk[5],
        'legal_prob_mass_per_token': metrics['legal_prob_mass_per_token'],
    }


class BoardProbe(nn.Module):
    def __init__(self, d_model, n_squares, probe_type):
        super().__init__()
        self.n_squares = n_squares
        if probe_type == 'linear':
            self.proj = nn.Linear(d_model, n_squares * 3)
        elif probe_type == 'mlp':
            self.proj = MLPProbe(
                d_model,
                n_squares * 3,
                hidden_dim=MLP_HIDDEN_DIM,
                n_layers=MLP_N_LAYERS,
                dropout=MLP_DROPOUT,
            )
        else:
            raise ValueError(f'Unknown probe_type: {probe_type}')

    def forward(self, x):
        logits = self.proj(x)
        return logits.view(*logits.shape[:-1], self.n_squares, 3)


class BoardProbeBank(nn.Module):
    def __init__(self, layers, d_model, n_squares, probe_type):
        super().__init__()
        self.absolute = nn.ModuleDict({
            str(layer): BoardProbe(d_model, n_squares, probe_type)
            for layer in layers
        })
        self.relative = nn.ModuleDict({
            str(layer): BoardProbe(d_model, n_squares, probe_type)
            for layer in layers
        })


def board_probe_loss(logits, labels):
    return F.cross_entropy(
        logits.reshape(-1, 3),
        labels.reshape(-1),
        ignore_index=LABEL_PAD,
    )


def board_metric_counts(logits, labels):
    preds = logits.argmax(dim=-1)
    valid = labels != LABEL_PAD
    return {
        'total': int(valid.sum().item()),
        'correct': int(((preds == labels) & valid).sum().item()),
    }


def merge_counts(target, source):
    for key, value in source.items():
        target[key] = target.get(key, 0) + int(value)


def counts_to_accuracy(counts):
    total = int(counts.get('total', 0))
    return {
        'accuracy': float(counts.get('correct', 0) / total) if total else float('nan'),
        'n_labels': total,
    }


@torch.no_grad()
def evaluate_board_probe_bank(probe_bank, games_raw):
    counts = {
        str(layer): {'absolute': {}, 'relative': {}}
        for layer in PROBE_LAYERS
    }
    context_encoder.eval()
    probe_bank.eval()

    with ActivationCache(context_encoder, PROBE_LAYERS) as cache:
        batches = _iter_probe_batches(
            games_raw,
            BOARD_SIZE,
            context_encoder.config.block_size,
            PROBE_BATCH_SIZE,
        )
        for x_cpu, y_abs_cpu, y_rel_cpu in tqdm(batches, desc='board probe val', leave=False):
            x = x_cpu.to(device)
            y_abs = y_abs_cpu.to(device)
            y_rel = y_rel_cpu.to(device)
            cache.clear()
            with amp_context(device, HEAD_PRECISION):
                context_encoder(x)
            for layer in PROBE_LAYERS:
                key = str(layer)
                acts = cache.activations[layer].to(device)
                abs_logits = probe_bank.absolute[key](acts)
                rel_logits = probe_bank.relative[key](acts)
                merge_counts(counts[key]['absolute'], board_metric_counts(abs_logits, y_abs))
                merge_counts(counts[key]['relative'], board_metric_counts(rel_logits, y_rel))

    return {
        key: {
            mode: counts_to_accuracy(mode_counts)
            for mode, mode_counts in layer_counts.items()
        }
        for key, layer_counts in counts.items()
    }


def load_probe_train_games_from_chunks(chunks, games_per_chunk):
    games = []
    per_chunk_counts = {}
    for chunk_path in chunks:
        chunk_games = load_chunk(str(chunk_path))
        selected = [list(game) for game in chunk_games if len(game) >= 2][:games_per_chunk]
        games.extend(selected)
        per_chunk_counts[Path(chunk_path).name] = len(selected)
    return games, per_chunk_counts


def train_board_probe_bank(probe_type):
    d_model = int(context_encoder.config.d_model)
    n_squares = BOARD_SIZE * BOARD_SIZE
    probe_bank = BoardProbeBank(PROBE_LAYERS, d_model, n_squares, probe_type).to(device)
    optimizer = torch.optim.AdamW(
        probe_bank.parameters(),
        lr=PROBE_LR,
        weight_decay=PROBE_WEIGHT_DECAY,
    )
    history = []
    context_encoder.eval()

    train_games, train_games_per_chunk = load_probe_train_games_from_chunks(
        train_chunks,
        PROBE_TRAIN_GAMES_PER_CHUNK,
    )
    print(
        f'{probe_type} board probe training games: '
        f'{len(train_games):,} from {len(train_chunks)} chunks '
        f'({PROBE_TRAIN_GAMES_PER_CHUNK}/chunk)'
    )

    with ActivationCache(context_encoder, PROBE_LAYERS) as cache:
        for epoch in range(1, PROBE_EPOCHS + 1):
            probe_bank.train()
            loss_sum = 0.0
            batch_count = 0
            batches = _iter_probe_batches(
                train_games,
                BOARD_SIZE,
                context_encoder.config.block_size,
                PROBE_BATCH_SIZE,
            )
            pbar = tqdm(batches, desc=f'{probe_type} board probes epoch {epoch}/{PROBE_EPOCHS}')
            for x_cpu, y_abs_cpu, y_rel_cpu in pbar:
                x = x_cpu.to(device)
                y_abs = y_abs_cpu.to(device)
                y_rel = y_rel_cpu.to(device)
                cache.clear()
                with torch.no_grad():
                    with amp_context(device, HEAD_PRECISION):
                        context_encoder(x)

                optimizer.zero_grad(set_to_none=True)
                loss = torch.zeros((), device=device)
                for layer in PROBE_LAYERS:
                    key = str(layer)
                    acts = cache.activations[layer].to(device)
                    loss = loss + board_probe_loss(probe_bank.absolute[key](acts), y_abs)
                    loss = loss + board_probe_loss(probe_bank.relative[key](acts), y_rel)
                loss = loss / (2 * len(PROBE_LAYERS))
                loss.backward()
                optimizer.step()

                loss_sum += float(loss.detach().cpu())
                batch_count += 1
                pbar.set_postfix(loss=f'{loss_sum / max(1, batch_count):.4f}')

            epoch_result = {
                'epoch': epoch,
                'train_loss': loss_sum / max(1, batch_count),
                'train_batches': batch_count,
            }
            history.append(epoch_result)
            print(f"{probe_type} board probes epoch={epoch} train_loss={epoch_result['train_loss']:.4f}")

    val_games = load_games_from_chunks(val_chunks, PROBE_VAL_GAMES)
    metrics = evaluate_board_probe_bank(probe_bank, val_games)
    result = {
        'probe_type': probe_type,
        'layers': list(PROBE_LAYERS),
        'train_chunks': [str(path) for path in train_chunks],
        'n_train_chunks': len(train_chunks),
        'probe_train_games_per_chunk': PROBE_TRAIN_GAMES_PER_CHUNK,
        'train_games_per_chunk': train_games_per_chunk,
        'n_train_games': len(train_games),
        'n_val_games': len(val_games),
        'epochs': PROBE_EPOCHS,
        'batch_size': PROBE_BATCH_SIZE,
        'lr': PROBE_LR,
        'weight_decay': PROBE_WEIGHT_DECAY,
        'hidden_dim': MLP_HIDDEN_DIM if probe_type == 'mlp' else None,
        'n_layers': MLP_N_LAYERS if probe_type == 'mlp' else None,
        'dropout': MLP_DROPOUT if probe_type == 'mlp' else None,
        'history': history,
        'metrics': metrics,
    }
    return probe_bank, result


def print_legal_metrics(label, metrics):
    compact = compact_legal_metrics(metrics)
    print(label)
    print(f"  top-1 legal          : {compact['top1_legal_per_token']:.2%}")
    print(f"  top-3 legal fraction : {compact['top3_legal_fraction']:.2%}")
    print(f"  top-5 legal fraction : {compact['top5_legal_fraction']:.2%}")


def print_board_table(label, result):
    print(label)
    print('layer | absolute | relative')
    print('----- | -------- | --------')
    for layer in result['layers']:
        metrics = result['metrics'][str(layer)]
        print(
            f"{layer:>5} | "
            f"{metrics['absolute']['accuracy']:>8.2%} | "
            f"{metrics['relative']['accuracy']:>8.2%}"
        )


print('Shared helpers ready.')

## 7. Linear Next-Move Head: Legal-Move Evaluation

In [ ]:
linear_head_model, linear_head_history = train_next_move_head('linear')
linear_legal_metrics = evaluate_legal_moves(
    model=linear_head_model,
    val_chunks=val_chunks,
    board_size=BOARD_SIZE,
    n_games=LEGAL_EVAL_GAMES,
    device=device,
    k_values=LEGAL_K_VALUES,
    batch_size=LEGAL_EVAL_BATCH_SIZE,
)

linear_head_path = SMOKE_EVAL_DIR / f'linear_next_move_head__{DRIVE_CKPT.stem}.pt'
torch.save(
    {
        'head': {key: value.detach().cpu() for key, value in linear_head_model.head.state_dict().items()},
        'head_type': 'linear_next_move',
        'checkpoint_path': str(DRIVE_CKPT),
        'train_chunks': [str(path) for path in train_chunks],
        'history': linear_head_history,
        'legal_metrics': jsonable(linear_legal_metrics),
    },
    linear_head_path,
)

SMOKE_RESULTS['next_move_legal']['linear'] = {
    'artifact_path': str(linear_head_path),
    'head_type': 'linear',
    'history': linear_head_history,
    'metrics': jsonable(linear_legal_metrics),
}
print_legal_metrics('Linear next-move head', linear_legal_metrics)
print('saved:', linear_head_path)

## 8. MLP Next-Move Head: Legal-Move Evaluation

In [ ]:
mlp_head_model, mlp_head_history = train_next_move_head('mlp')
mlp_legal_metrics = evaluate_legal_moves(
    model=mlp_head_model,
    val_chunks=val_chunks,
    board_size=BOARD_SIZE,
    n_games=LEGAL_EVAL_GAMES,
    device=device,
    k_values=LEGAL_K_VALUES,
    batch_size=LEGAL_EVAL_BATCH_SIZE,
)

mlp_head_path = SMOKE_EVAL_DIR / f'mlp_next_move_head__{DRIVE_CKPT.stem}.pt'
torch.save(
    {
        'head': {key: value.detach().cpu() for key, value in mlp_head_model.head.state_dict().items()},
        'head_type': 'mlp_next_move',
        'checkpoint_path': str(DRIVE_CKPT),
        'hidden_dim': MLP_HIDDEN_DIM,
        'n_layers': MLP_N_LAYERS,
        'dropout': MLP_DROPOUT,
        'train_chunks': [str(path) for path in train_chunks],
        'history': mlp_head_history,
        'legal_metrics': jsonable(mlp_legal_metrics),
    },
    mlp_head_path,
)

SMOKE_RESULTS['next_move_legal']['mlp'] = {
    'artifact_path': str(mlp_head_path),
    'head_type': 'mlp',
    'hidden_dim': MLP_HIDDEN_DIM,
    'n_layers': MLP_N_LAYERS,
    'dropout': MLP_DROPOUT,
    'history': mlp_head_history,
    'metrics': jsonable(mlp_legal_metrics),
}
print_legal_metrics('MLP next-move head', mlp_legal_metrics)
print('saved:', mlp_head_path)

## 9. Linear Board-State Probes: Absolute And Relative Per Layer

In [ ]:
linear_probe_bank, linear_probe_results = train_board_probe_bank('linear')
linear_probe_path = SMOKE_EVAL_DIR / f'linear_board_probes__{DRIVE_CKPT.stem}.pt'
torch.save(
    {
        'probe_bank': {
            key: value.detach().cpu()
            for key, value in linear_probe_bank.state_dict().items()
        },
        'results': jsonable(linear_probe_results),
        'checkpoint_path': str(DRIVE_CKPT),
    },
    linear_probe_path,
)

SMOKE_RESULTS['board_state']['linear'] = {
    'artifact_path': str(linear_probe_path),
    'results': jsonable(linear_probe_results),
}
print_board_table('Linear board-state probes', linear_probe_results)
print('saved:', linear_probe_path)

## 10. MLP Board-State Probes: Absolute And Relative Per Layer

In [ ]:
mlp_probe_bank, mlp_probe_results = train_board_probe_bank('mlp')
mlp_probe_path = SMOKE_EVAL_DIR / f'mlp_board_probes__{DRIVE_CKPT.stem}.pt'
torch.save(
    {
        'probe_bank': {
            key: value.detach().cpu()
            for key, value in mlp_probe_bank.state_dict().items()
        },
        'results': jsonable(mlp_probe_results),
        'checkpoint_path': str(DRIVE_CKPT),
        'hidden_dim': MLP_HIDDEN_DIM,
        'n_layers': MLP_N_LAYERS,
        'dropout': MLP_DROPOUT,
    },
    mlp_probe_path,
)

SMOKE_RESULTS['board_state']['mlp'] = {
    'artifact_path': str(mlp_probe_path),
    'results': jsonable(mlp_probe_results),
}
print_board_table('MLP board-state probes', mlp_probe_results)
print('saved:', mlp_probe_path)

## 10b. Matched-Conditions Board Probes (Diagnostic)

The standard probes above average over EVERY position of full games — including positions the objective never supervised. This diagnostic trains the same per-layer probes on hidden states sampled exactly under the training conditions of prefix-based objectives: **pass-free prefixes, t in [4, 44], hidden state at position t-1 only**. A large gap vs the standard tables separates 'board state concentrated at trained readout positions' from 'no board state anywhere'.

In [ ]:
import importlib
import othello_research.probes.matched_prefix_probe as mpp
importlib.reload(mpp)

MATCHED_T_MIN = 4
MATCHED_T_MAX = 44
MATCHED_PREFIXES_PER_GAME = 4

matched_train_games, _ = load_probe_train_games_from_chunks(
    train_chunks, PROBE_TRAIN_GAMES_PER_CHUNK
)
matched_val_games = load_games_from_chunks(val_chunks, PROBE_VAL_GAMES)

matched_probe_results = {}
for probe_type in ('linear', 'mlp'):
    result = mpp.train_matched_probe_bank(
        context_encoder,
        matched_train_games,
        matched_val_games,
        layers=PROBE_LAYERS,
        device=device,
        probe_type=probe_type,
        hidden_dim=MLP_HIDDEN_DIM,
        n_probe_layers=MLP_N_LAYERS,
        dropout=MLP_DROPOUT,
        epochs=PROBE_EPOCHS,
        batch_size=PROBE_BATCH_SIZE,
        lr=PROBE_LR,
        weight_decay=PROBE_WEIGHT_DECAY,
        precision=HEAD_PRECISION,
        board_size=BOARD_SIZE,
        t_min=MATCHED_T_MIN,
        t_max=MATCHED_T_MAX,
        prefixes_per_game=MATCHED_PREFIXES_PER_GAME,
        seed=42,
    )
    matched_probe_results[probe_type] = result
    mpp.print_matched_table(
        f'{probe_type.upper()} matched-conditions board probes '
        f'(pass-free, last position, t in [{MATCHED_T_MIN},{MATCHED_T_MAX}])',
        result,
    )
    print()

SMOKE_RESULTS['board_state']['matched'] = jsonable(matched_probe_results)


## 11. Consolidated Summary

In [ ]:
def pct(value):
    return 'n/a' if value is None else f'{100 * float(value):.2f}%'


def topk_metric(metrics, k):
    values = metrics['topk_legal']
    return values.get(k, values.get(str(k)))


def best_board_layer(result, mode):
    rows = []
    for layer in result['layers']:
        accuracy = result['metrics'][str(layer)][mode]['accuracy']
        rows.append((float(accuracy), int(layer)))
    return max(rows)


linear_legal = SMOKE_RESULTS['next_move_legal']['linear']['metrics']
mlp_legal = SMOKE_RESULTS['next_move_legal']['mlp']['metrics']
linear_board = SMOKE_RESULTS['board_state']['linear']['results']
mlp_board = SMOKE_RESULTS['board_state']['mlp']['results']

linear_abs_best, linear_abs_layer = best_board_layer(linear_board, 'absolute')
linear_rel_best, linear_rel_layer = best_board_layer(linear_board, 'relative')
mlp_abs_best, mlp_abs_layer = best_board_layer(mlp_board, 'absolute')
mlp_rel_best, mlp_rel_layer = best_board_layer(mlp_board, 'relative')

lines = [
    f'# JEPA Smoke Evaluation: {JEPA_RUN_NAME}',
    '',
    f'- **Checkpoint:** `{DRIVE_CKPT.name}`',
    f'- **Objective:** `{objective_name}`',
    f'- **Checkpoint step:** `{ckpt.get("step")}`',
    f'- **Games seen:** `{ckpt.get("games_seen", 0):,}`',
    f'- **Downstream training chunks:** `{EVAL_TRAIN_CHUNKS}`',
    f'- **Board probe training games:** `{PROBE_TRAIN_GAMES_PER_CHUNK * len(train_chunks):,}` (`{PROBE_TRAIN_GAMES_PER_CHUNK}` per chunk)',
    f'- **Validation split begins after pretraining chunk:** `{N_TRAIN_CHUNKS}`',
    '',
    '## Next-Move Heads: Legal-Move Evaluation',
    '',
    'The heads are trained with next-move cross-entropy. These metrics evaluate whether their predictions are legal, not whether they match the sampled continuation.',
    '',
    '| Head | Top-1 legal | Top-3 legal fraction | Top-5 legal fraction | Legal probability mass |',
    '|---|---:|---:|---:|---:|',
    (
        f"| Linear | {pct(linear_legal['top1_legal_per_token'])} | "
        f"{pct(topk_metric(linear_legal, 3))} | {pct(topk_metric(linear_legal, 5))} | "
        f"{pct(linear_legal['legal_prob_mass_per_token'])} |"
    ),
    (
        f"| MLP | {pct(mlp_legal['top1_legal_per_token'])} | "
        f"{pct(topk_metric(mlp_legal, 3))} | {pct(topk_metric(mlp_legal, 5))} | "
        f"{pct(mlp_legal['legal_prob_mass_per_token'])} |"
    ),
    '',
    '## Board-State Probes: Best Validation Layers',
    '',
    '| Probe | Best absolute layer | Absolute accuracy | Best relative layer | Relative accuracy |',
    '|---|---:|---:|---:|---:|',
    f'| Linear | L{linear_abs_layer} | {pct(linear_abs_best)} | L{linear_rel_layer} | {pct(linear_rel_best)} |',
    f'| MLP | L{mlp_abs_layer} | {pct(mlp_abs_best)} | L{mlp_rel_layer} | {pct(mlp_rel_best)} |',
    '',
    '## Linear Board-State Probe: Per-Layer Validation Accuracy',
    '',
    '| Layer | Absolute accuracy | Relative accuracy |',
    '|---:|---:|---:|',
]

for layer in linear_board['layers']:
    metrics = linear_board['metrics'][str(layer)]
    lines.append(
        f"| L{layer} | {pct(metrics['absolute']['accuracy'])} | "
        f"{pct(metrics['relative']['accuracy'])} |"
    )

lines += [
    '',
    '## MLP Board-State Probe: Per-Layer Validation Accuracy',
    '',
    '| Layer | Absolute accuracy | Relative accuracy |',
    '|---:|---:|---:|',
]

for layer in mlp_board['layers']:
    metrics = mlp_board['metrics'][str(layer)]
    lines.append(
        f"| L{layer} | {pct(metrics['absolute']['accuracy'])} | "
        f"{pct(metrics['relative']['accuracy'])} |"
    )

if 'matched_probe_results' in globals() and matched_probe_results:
    lines += [
        '',
        '## Matched-Conditions Board Probes (pass-free, last position, '
        f"t in [{matched_probe_results['mlp']['t_min']},"
        f"{matched_probe_results['mlp']['t_max']}])",
        '',
        'Probes trained only on hidden states at position t-1 of pass-free '
        'prefixes — the exact conditions prefix-based objectives supervise. '
        'Compare against the all-position tables above.',
        '',
        '| Layer | Linear abs | Linear rel | MLP abs | MLP rel |',
        '|---:|---:|---:|---:|---:|',
    ]
    for layer in matched_probe_results['mlp']['layers']:
        lin = matched_probe_results['linear']['metrics'][str(layer)]
        mlp = matched_probe_results['mlp']['metrics'][str(layer)]
        lines.append(
            f"| L{layer} | {pct(lin['absolute']['accuracy'])} "
            f"| {pct(lin['relative']['accuracy'])} "
            f"| {pct(mlp['absolute']['accuracy'])} "
            f"| {pct(mlp['relative']['accuracy'])} |"
        )

lines += [
    '',
    '## Artifacts',
    '',
    f"- Linear next-move head: `{SMOKE_RESULTS['next_move_legal']['linear']['artifact_path']}`",
    f"- MLP next-move head: `{SMOKE_RESULTS['next_move_legal']['mlp']['artifact_path']}`",
    f"- Linear board probes: `{SMOKE_RESULTS['board_state']['linear']['artifact_path']}`",
    f"- MLP board probes: `{SMOKE_RESULTS['board_state']['mlp']['artifact_path']}`",
]

summary_markdown = '\n'.join(lines) + '\n'
summary_path = SMOKE_EVAL_DIR / f'smoke_eval_summary__{DRIVE_CKPT.stem}.md'
results_path = SMOKE_EVAL_DIR / f'smoke_eval_results__{DRIVE_CKPT.stem}.json'

summary_path.write_text(summary_markdown, encoding='utf-8')
results_path.write_text(json.dumps(jsonable(SMOKE_RESULTS), indent=2), encoding='utf-8')

display(Markdown(summary_markdown))
print('saved summary:', summary_path)
print('saved results:', results_path)

In [ ]:
from google.colab import runtime
runtime.unassign()